In [ ]:
import numpy as np
import pandas as pd
import pickle
import networkx as nx
import community as community_louvain
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def load_and_filter(embeddings_path):
    print("📥 Загрузка и фильтрация токенов...")
    with open(embeddings_path, 'rb') as f:
        emb_dict = pickle.load(f)

    valid = []
    for key, data in emb_dict.items():
        if data['form'] == '#NULL' or data['sem_class'] == '_': continue
        valid.append({
            'key': key,
            'lemma': data['lemma'],
            'form': data['form'],
            'emb': np.asarray(data['embedding'], dtype=np.float32)
        })
    print(f"   Отобрано {len(valid)} валидных токенов.")
    return valid

def build_prototypes(valid_tokens, lemma_sim_thresh=0.60):
    print("Агрегация прототипов (с разделением полисемии)...")
    lemma_idx = defaultdict(list)
    for i, tok in enumerate(valid_tokens):
        lemma_idx[tok['lemma']].append(i)

    prototypes = []
    for lem, indices in tqdm(lemma_idx.items(), desc="Леммы"):
        if len(indices) == 1:
            prototypes.append({'lemma': lem, 'emb': valid_tokens[indices[0]]['emb'], 'orig_indices': [indices[0]]})
            continue

        X_lem = np.stack([valid_tokens[i]['emb'] for i in indices])
        sims = cosine_similarity(X_lem)
        np.fill_diagonal(sims, 0.0)

        assigned = set()
        for i in range(len(indices)):
            if i in assigned: continue
            cluster = [i]
            assigned.add(i)
            for j in range(i+1, len(indices)):
                if j not in assigned and sims[i, j] > lemma_sim_thresh:
                    cluster.append(j)
                    assigned.add(j)
            # Усредняем прототип смысла
            mean_emb = np.mean(X_lem[cluster], axis=0)
            mean_emb /= (np.linalg.norm(mean_emb) + 1e-8)
            prototypes.append({
                'lemma': lem,
                'emb': mean_emb,
                'orig_indices': [indices[idx] for idx in cluster]
            })

    print(f"   Создано {len(prototypes)} стабильных прототипов.")
    return prototypes

In [ ]:
def build_mutual_knn_graph(prototypes, k=30, sim_thresh=0.45):
    print("Построение графа транзитивного сходства...")
    X = np.stack([p['emb'] for p in prototypes])

    # k-NN поиск
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute').fit(X)
    dists, idxs = nn.kneighbors(X)
    dists, idxs = dists[:, 1:], idxs[:, 1:]  # убираем самого себя

    G = nx.Graph()
    G.add_nodes_from(range(len(prototypes)))

    edges_added = 0
    for i in range(len(prototypes)):
        for pos, j in enumerate(idxs[i]):
            sim = 1.0 - dists[i, pos]
            if sim < sim_thresh: continue
            # Проверка взаимности: j должен видеть i в своём k-списке
            if i in idxs[j]:
                G.add_edge(i, j, weight=sim)
                edges_added += 1

    # Удаляем изолированные узлы
    G.remove_nodes_from(list(nx.isolates(G)))
    print(f"   Граф: {G.number_of_nodes()} узлов, {edges_added} рёбер.")
    return G

In [ ]:
def detect_communities_hierarchy(G, thresholds=[0.65, 0.55, 0.45, 0.35]):
    print("Извлечение иерархии сообществ...")
    levels = {}

    for t in thresholds:
        G_t = G.copy()
        # Фильтрация слабых рёбер
        G_t.remove_edges_from([(u,v) for u,v,d in G_t.edges(data=True) if d['weight'] < t])
        G_t.remove_nodes_from(list(nx.isolates(G_t)))

        if G_t.number_of_nodes() < 2:
            continue

        try:
            communities = nx.algorithms.community.louvain_communities(G_t, weight='weight', seed=42)
            communities = nx.community.louvain_communities(G, weight='weight', resolution=0.3, seed=42)
        except AttributeError:
            communities = nx.algorithms.community.greedy_modularity_communities(G_t, weight='weight')

        # Конвертируем список множеств в словарь {node_id: community_id}
        partition = {}
        for cid, comm_nodes in enumerate(communities):
            for node in comm_nodes:
                partition[node] = cid

        levels[t] = partition
        print(f"   threshold={t:.2f} → {len(communities)} сообществ")

    return levels

In [ ]:
def map_to_tokens_and_export(valid_tokens, prototypes, levels, output_path):
    print("Маппинг на исходные токены и экспорт...")
    tok_to_proto = {}
    for p_idx, proto in enumerate(prototypes):
        for tok_idx in proto['orig_indices']:
            tok_to_proto[tok_idx] = p_idx

    rows = []
    for tok_idx, tok_data in enumerate(valid_tokens):
        if tok_idx not in tok_to_proto: continue
        p_idx = tok_to_proto[tok_idx]

        row = {
            'token_key': str(tok_data['key']),
            'lemma': tok_data['lemma'],
            'form': tok_data['form']
        }
        for t, partition in levels.items():
            row[f'cluster_t{t}'] = partition.get(p_idx, -1)
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"   Онтология сохранена: {output_path}")
    return df

In [ ]:
EMB_PATH = '/content/drive/MyDrive/Master_thesis/qwen35_4b_middle_embeddings.pkl'
OUT_PATH = 'auto_ontology_hierarchy.csv'

tokens = load_and_filter(EMB_PATH)
protos = build_prototypes(tokens, lemma_sim_thresh=0.58)
G = build_mutual_knn_graph(protos, k=25, sim_thresh=0.42)

In [ ]:
levels = detect_communities_hierarchy(G, thresholds=[0.65, 0.55, 0.45, 0.35])
ontology_df = map_to_tokens_and_export(tokens, protos, levels, OUT_PATH)

print("\nПример структуры онтологии (первые 5 строк):")
print(ontology_df.head())